# Template matching demo

In [ ]:
%matplotlib inline
import time
from IPython import display
import ipywidgets as ipw

import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
!wget -nc https://www.dais.unive.it/~bergamasco/teachingfiles/cvstuff/galaxian.mp4
!wget -nc https://www.dais.unive.it/~bergamasco/teachingfiles/cvstuff/template.png

In [ ]:
cap = cv.VideoCapture("galaxian.mp4")
cap.set(cv.CAP_PROP_POS_FRAMES, 750)

T = cv.imread("template.png", cv.IMREAD_GRAYSCALE ).astype(np.float32)
T = T - np.mean(T)
plt.figure( figsize=(5,5))
plt.imshow(T, interpolation="none", cmap="gray")
plt.title("Template")

In [ ]:
wIm = ipw.Image()
wTx = ipw.Label(value='Hello World'  )

display.display(wTx)
display.display(wIm)


while cap.isOpened():

    ret,frame = cap.read()

    img_g = cv.cvtColor( frame, cv.COLOR_BGR2GRAY )
    img_g = img_g.astype(np.float32)
    img_g = img_g - np.mean(img_g)

    conv = cv.filter2D( img_g, -1, T ) # OpenCV filter2D computes the correlation between img_g and T
    maxval = np.amax( conv )

    if maxval>5E6: # Correlation threshold
        maxpos = np.unravel_index( np.argmax( conv ), conv.shape )
    else:
        maxpos=[-999,-999]

    cv.rectangle( frame, (maxpos[1]-15, maxpos[0]-15),
                         (maxpos[1]+15, maxpos[0]+15),
                         (0,0,255), 4 )

    #_, ienc = cv.imencode(".jpg", frame)
    _, ienc = cv.imencode(".jpg", conv*255//np.amax(conv) )

    wIm.value = ienc.tobytes()
    wTx.value = "Ship position %d-%d  (peak val: %3.2f)"%(maxpos[1],maxpos[0],maxval)
    time.sleep(1/25)

In [ ]:
!rm galaxian.mp4 template.png